In [ ]:
import seaborn as sns
import pandas as pd
import matplotlib.pyplot as plt
import os
import pyfmreader as pyfm
import pyfmreader.ps_nex.parseTDMS as tdms
import numpy as np
import pyfmreader.ps_nex.loadpsnexMaps as maps
import glob


### Load Psnex_map_ folder that contains force curves and/or a CSV. file of heights in a 2D array csv file
Load a PSNEX map with either 2D array of heights, if no CSV file is found, 

In [ ]:
# Define map direcotory path
file_path = '/Users/evillz/Data/psnex_map___2025.03.12_18.23.11.20'
file_path = '/Volumes/SSK_Drive/2026_map_ctc44_3d/2026_04_23/sample1/maps/psnex_map___2026.04.23_15.39.46.49'
# file_path = '/Users/evillz/Data/CTC44/2025_12_03/psnex_map___2025.03.12_19.02.17.49'

# file_path = '/Users/evillz/Data/CTC44/CTC_chirp_yogesh/psnex_map___2025.03.21_19.44.03.14'
# file_path = '/Users/evillz/Data/CTC44/2025_12_03/psnex_map___2025.03.12_18.38.42.45'



# Find CSV files in the specified directory
csv_files = maps.find_csv_files(file_path)

# Get the first CSV file found (if any)
if csv_files:
    map_path_csv = csv_files[0]

# load Map parameters 
params = maps.get_map_parameters(file_path)

In [ ]:
from pyfmreader.ps_nex.parseTDMS import grab_tdms
first_file, tdms_files = grab_tdms(file_path)
print (len(tdms_files))

In [ ]:
force_curve = pyfm.loadfile(first_file)

metadata = force_curve.filemetadata

In [ ]:
from pyfmreader.plotting import load_fcurve_metadata


metadata_2, piezo_um, deflection_zero_N, time_s = load_fcurve_metadata(first_file)

In [ ]:
if csv_files:
    map_test_um_zeroed, x_axis, y_axis = maps.load_2d_map_file_csv (map_path_csv, params)

df_map, map_x_pix, map_y_pix , params = maps.load_map_file_square_tdms(file_path)

# # need


_ = maps.create_heatmap(df_map, params, value_key='z_height_um_zero', annot=True, correct_indices=True)




In [ ]:
print (map_x_pix, map_y_pix)
print (df_map.size)

In [ ]:
# check map for map integrity

erro_bool, _ = maps.check_map_indice_integrity(df_map, map_x_pix, map_y_pix)

In [ ]:
# generate x,y positions for map plotting
x_pos_1d, y_pos_1d, curve_index = maps.generate_xy_map_positions(map_x_pix=map_x_pix, map_y_pix=map_y_pix)   

In [ ]:
# Check if input is a dictionary from load_map_file_square_tdms
value_key="z_height_um_zero"
zmin = None
zmax = None
flipAxis = True

if isinstance(df_map, dict):
    print("Using dictionary from load_map_file_square_tdms")

    # noticed that some of the maps have some errors in the curve index. Some 
    # curves have duplicated index or x or Y positions
    

    # Create a 2D array from the dictionary data
    # x_indices = df_map['x_index']
    # y_indices = df_map['y_index']
    x_indices = x_pos_1d
    y_indices = y_pos_1d
    # Use the specified key, fallback to 'z_height_um_zero' if not present
    if value_key in df_map:
        z_values = df_map[value_key]
    elif 'z_height_um_zero' in df_map:
        z_values = df_map['z_height_um_zero']
    else:
        z_values = df_map['z_height_um']

    # Get dimensions from params or from max indices
    map_x_pix = params['map_x_pix'] if params and 'map_x_pix' in params else int(np.max(x_indices)) + 1
    map_y_pix = params['map_y_pix'] if params and 'map_y_pix' in params else int(np.max(y_indices)) + 1

    # Create empty 2D array and fill with values
    map_test_um_zeroed = np.zeros((map_y_pix, map_x_pix), dtype=np.float64)

    # map data parsing and filling the 2D array
    # print (f'type map_test_um_zeroed Before: {type(map_test_um_zeroed)}')
    for i in range(len(z_values)):
        x_idx = int(x_indices[i])
        y_idx = int(y_indices[i])
        # print (map_x_pix, map_y_pix)
        if 0 <= x_idx < map_x_pix and 0 <= y_idx < map_y_pix:
            # print(f'{i}: x_idx: {x_idx}, y_idx: {y_idx}, z_value: {z_values[i]}')
            map_test_um_zeroed[y_idx, x_idx] = z_values[i]
    
    # get the last column and move it to the first column. Remove this if fixed in the future -LDV
    map_test_um_zeroed = np.roll(map_test_um_zeroed, shift=1, axis=1)  

    # Convert to DataFrame for consistency with the rest of the function
    df_map_test_um_zeroed = pd.DataFrame(map_test_um_zeroed, dtype=np.float64)
    map_path = "Dictionary Input"  # Default path name when using Dictionary
    print (f'type df_map_test_um_zeroed After: {type(df_map_test_um_zeroed)}')
    # print (df_map_test_um_zeroed)

elif isinstance(df_map, pd.DataFrame):
    # Assume df_map is a DataFrame with at least x_index, y_index, and value_key columns
        print("Using provided DataFrame for heatmap")
        correct_indices = True
        lastColFirst = True  # remove when fixed: only last column in image must be moved to first column
        x_indices = df_map['x_index'].values
        y_indices = df_map['y_index'].values

        # Get dimensions from params or from max indices
        map_x_pix = params['map_x_pix'] if params and 'map_x_pix' in params else int(np.max(x_indices)) + 1
        map_y_pix = params['map_y_pix'] if params and 'map_y_pix' in params else int(np.max(y_indices)) + 1

        print(f'Map dimensions: {map_x_pix} x {map_y_pix}')
        if correct_indices:
            bool_error, _ = maps.check_map_indice_integrity(df_map, map_x_pix, map_y_pix)
            if bool_error:
                x_pos_1d, y_pos_1d, curve_index = maps.generate_xy_map_positions(map_x_pix=map_x_pix, map_y_pix=map_y_pix)
                x_indices = x_pos_1d
                y_indices = y_pos_1d
                print('Map indices were regenerated due to integrity error')
                df_map['curve_index'] = curve_index
                df_map['x_index'] = x_indices
                df_map['y_index'] = y_indices

        # Use the specified key, fallback to 'z_height_um_zero' if not present
        if value_key in df_map:
            z_values = df_map[value_key].values
        else:
            z_values = df_map['z_height_um'].values

        # Create empty 2D array and fill with values
        map_test_um_zeroed = np.zeros((map_y_pix, map_x_pix), dtype=np.float64)
        for i in range(len(z_values)):
            x_idx = int(x_indices[i])
            y_idx = int(y_indices[i])
            if 0 <= x_idx < map_x_pix and 0 <= y_idx < map_y_pix:
                if np.isnan(z_values[i]):
                    map_test_um_zeroed[y_idx, x_idx] = 0
                else:
                    map_test_um_zeroed[y_idx, x_idx] = z_values[i]

        if lastColFirst:
            map_test_um_zeroed = np.roll(map_test_um_zeroed, shift=1, axis=1)

        df_map_test_um_zeroed = pd.DataFrame(map_test_um_zeroed, dtype=np.float64)
        map_path = "DataFrame Input"  # Default path name when using DataFrame
else:
    # Assume df_map is a path to a CSV file
    map_path = df_map
    print(f'Loading map from file: {map_path}')

##% mirror flip the 2D array along the vertical and horizontal axis %##
# df_map_test_um_zeroed = df_map_test_um_zeroed.iloc[:, ::-1]

# Default parameters
defaults = {
    'z_sens_um': 6,
    'x_sens_um': 5.685,
    'y_sens_um': 3.960,
    'map_x_step': 0.14,
    'map_y_step': 0.14,
    'map_x_pix': df_map_test_um_zeroed.shape[0],
    'map_y_pix': df_map_test_um_zeroed.shape[1]
}

# Override defaults with params if provided
if params is not None:
    for k in defaults:
        if k in params:
            defaults[k] = params[k]

# z_sens_um = defaults['z_sens_um']
x_sens_um = defaults['x_sens_um']
y_sens_um = defaults['y_sens_um']
dx_v = defaults['map_x_step']
dy_v = defaults['map_y_step']
map_x_pix = defaults['map_x_pix']
map_y_pix = defaults['map_y_pix']

# Generate axis values
x_axis = (np.arange(map_x_pix) * dx_v * x_sens_um).round(1)
y_axis = (np.arange(map_y_pix) * dy_v * y_sens_um).round(1)

# Define the z-axis range for the colormap
z_min = zmin if zmin is not None else df_map_test_um_zeroed.min().min()
z_max = zmax if zmax is not None else df_map_test_um_zeroed.max().max()
vmin = z_min
vmax = z_max

# remove when fixed: only last column in image must be moved to first column
# df_map_test_um_zeroed = df_map_test_um_zeroed.iloc[:, [0, -1]] + df_map_test_um_zeroed.iloc[:, :-1] 

fig, ax = plt.subplots(figsize=(20, 10))
a = sns.heatmap(df_map_test_um_zeroed, annot=df_map_test_um_zeroed,xticklabels=x_axis.round(2), yticklabels=y_axis.round(2), ax=ax, cmap="YlOrBr", vmin=vmin, vmax=vmax)
a.invert_yaxis()
xy_axis = -1 if flipAxis else 1

a.set_aspect((x_sens_um / y_sens_um) ** xy_axis)

print(f'y_sens_um: {y_sens_um}, x_sens_um: {x_sens_um}')

a.set_xlabel('X axis (um)', fontsize=20)
a.set_ylabel('Y axis (um)', fontsize=20)
colorbar = a.collections[0].colorbar
colorbar.set_label('Z axis (um)', fontsize=18)
colorbar.ax.yaxis.label.set_rotation(90)
_ = plt.xticks(ticks=np.arange(0, len(x_axis), 5), labels=x_axis[::5].round(2), rotation=45, fontsize=18)
_ = plt.yticks(ticks=np.arange(0, len(y_axis), 5), labels=y_axis[::5].round(2), rotation=0, fontsize=18)
colorbar.ax.tick_params(labelsize=20)
filename = os.path.basename(map_path)
a.set_title(f'Heatmap: {filename}', fontsize=20)
ax.minorticks_on()
ax.tick_params(axis='both', which='minor', length=4, color='black')
fig.patch.set_alpha(0.0)

In [ ]:
map_path = '/Users/evillz/Data/CTC44/2025_12_03/psnex_map___2025.03.12_18.05.10.71'

# Process the map and create heatmap
map_test, map_test_csv, x_axis, y_axis, df_maps, params = maps.process_map_and_create_heatmap(map_path)

# Create the output TIFF filename
tiff_filename_csv = os.path.join(map_path, os.path.basename(map_path) +"csv" + ".tiff")
print(map_path)
# save tiff file 
_ = maps.save_map_as_tiff (map_test_csv, px_um_x=params['map_x_step'], px_um_y= params['map_y_step'], out_path=tiff_filename_csv)

# Create the output TIFF filename
tiff_filename = os.path.join(map_path, os.path.basename(map_path) +"tdms" + ".tiff")
print(map_path)
print (f'type map_test : {type(map_test)}')
# save tiff file 
if map_test is not None:

    _ = maps.save_map_as_tiff (map_test, px_um_x=params['map_x_step'], px_um_y= params['map_y_step'], out_path=tiff_filename)



In [ ]:
print(type(df_maps))
print("df_maps is a dictionary with keys:")
print(df_maps.keys())

# for key in df_maps.keys():

#     if key == 'filepath':
#         continue  # Skip the filepath key
#     if np.isnan(df_maps[key]).any():
#         print(f"NaN values found in '{key}' column.")
#     else:
#         print(f"No NaN values in '{key}' column.")    


In [ ]:
# Create the output TIFF filename
tiff_filename_csv = os.path.join(map_path, os.path.basename(map_path) +"csv" + ".tiff")
print(map_path)
# save tiff file 
_ = maps.save_map_as_tiff (map_test_csv, px_um_x=params['map_x_step'], px_um_y= params['map_y_step'], out_path=tiff_filename_csv)

# Create the output TIFF filename
tiff_filename = os.path.join(map_path, os.path.basename(map_path) +"tdms" + ".tiff")
print(map_path)
print(type(map_test))

# save tiff file 
_ = maps.save_map_as_tiff (map_test, px_um_x=params['map_x_step'], px_um_y= params['map_y_step'], out_path=tiff_filename)


In [ ]:
# # print params
# print(params)
# params['map_x_step']

In [ ]:
map_path = '/Users/evillz/Data/CTC44/2025_12_03/psnex_map___2025.03.12_18.38.42.45' # CTC 44 PFQNM. THIS
# Process the map and create heatmap
map_test, map_test_csv, x_axis, y_axis, df_maps, params = maps.process_map_and_create_heatmap(map_path, zmax=35)

# # Create the output TIFF filename
# tiff_filename_csv = os.path.join(map_path, os.path.basename(map_path) +"csv" + ".tiff")
# print(map_path)
# # save tiff file 
# _ = maps.save_map_as_tiff (map_test_csv, px_um_x=params['map_x_step'], px_um_y= params['map_y_step'], out_path=tiff_filename_csv)

# # Create the output TIFF filename
# tiff_filename = os.path.join(map_path, os.path.basename(map_path) +"tdms" + ".tiff")
# print(map_path)

# # save tiff file 
# _ = maps.save_map_as_tiff (map_test, px_um_x=params['map_x_step'], px_um_y= params['map_y_step'], out_path=tiff_filename)


In [ ]:
_ = maps.compute_time_difference("/Users/evillz/Data/CTC44/2025_12_03/psnex_map___2025.03.12_17.51.51.34")

_ = maps.compute_time_difference('/Users/evillz/Data/psnex_map___2025.03.12_18.23.11.20.map')

In [ ]:
directory = '/Users/evillz/Data/'
pattern = os.path.join(directory, 'psnex_map_*')
# Find only directories that match the pattern
files = [f for f in glob.glob(pattern) if os.path.isdir(f) and not (f.endswith('.zip') or f.endswith('.csv'))]
print(files)

In [ ]:
# for map_path in files:
#     # map_path = '/Users/evillz/Data/CTC44/2025_12_03/psnex_map___2025.03.12_18.38.42.45' # CTC 44 PFQNM. THIS
#     # Process the map and create heatmap
#     print ("Processing: ", map_path)
#     map_test, map_test_csv, x_axis, y_axis, df_maps, params = maps.process_map_and_create_heatmap(map_path)

#     # Create the output TIFF filename
#     tiff_filename_csv = os.path.join(map_path, os.path.basename(map_path) +"csv" + ".tiff")
#     print(map_path)
#     # save tiff file 
#     if map_test_csv is not None:
#         _ = maps.save_map_as_tiff (map_test_csv, px_um_x=params['map_x_step'], px_um_y= params['map_y_step'], out_path=tiff_filename_csv)

#     # Create the output TIFF filename
#     tiff_filename = os.path.join(map_path, os.path.basename(map_path) +"tdms" + ".tiff")
#     print(map_path)

#     # save tiff file 
#     if map_test is not None:
#         _ = maps.save_map_as_tiff (map_test, px_um_x=params['map_x_step'], px_um_y= params['map_y_step'], out_path=tiff_filename)

In [ ]:
map_path = '/Users/evillz/Data/CTC44/CTC_chirp_yogesh/psnex_map___2025.03.21_19.44.03.14'
print ("Processing: ", map_path)
map_test, map_test_csv, x_axis, y_axis, df_maps, params = maps.process_map_and_create_heatmap(map_path)

# Create the output TIFF filename
tiff_filename_csv = os.path.join(map_path, os.path.basename(map_path) +"csv" + ".tiff")
print(map_path)
# save tiff file 
if map_test_csv is not None:
    _ = maps.save_map_as_tiff (map_test_csv, px_um_x=params['map_x_step'], px_um_y= params['map_y_step'], out_path=tiff_filename_csv)

# Create the output TIFF filename
tiff_filename = os.path.join(map_path, os.path.basename(map_path) +"tdms" + ".tiff")
print(map_path)

# save tiff file 
if map_test is not None:
    _ = maps.save_map_as_tiff (map_test, px_um_x=params['map_x_step'], px_um_y= params['map_y_step'], out_path=tiff_filename)

In [ ]:
import seaborn as sns
import pandas as pd
import matplotlib.pyplot as plt
import os
import pyfmreader as pyfm
import pyfmreader.ps_nex.parseTDMS as tdms
import numpy as np
import pyfmreader.ps_nex.loadpsnexMaps as maps
import glob


In [ ]:
# Example usage
root_dir = '/Users/evillz/Data/'
tiff_folder = '/Users/evillz/Data/all_tiff_new_v2'
psnex_map_folders = maps.find_psnex_map_folders(root_dir)
print(f"Found {len(psnex_map_folders)} psnex_map_ folders:")
# for folder in psnex_map_folders:
#     print(f"  - {folder}")
# os.path.basename(map_path)

In [ ]:
# for map_path in psnex_map_folders[:]:
#     print(f"  - {map_path}")
#     # map_path = '/Users/evillz/Data/CTC44/CTC_chirp_yogesh/psnex_map___2025.03.21_19.44.03.14'
#     print ("Processing: ", map_path)
#     try: 
#         map_test, map_test_csv, x_axis, y_axis, df_maps, params = maps.process_map_and_create_heatmap(map_path, correct_indices=True)

#         # Create the output TIFF filename
#         tiff_filename_csv = os.path.join(map_path, tiff_folder +"csv_test_w_correction_v2" + ".tiff")
#         print(map_path)
#         # save tiff file 
#         if map_test_csv is not None:
#             _ = maps.save_map_as_tiff (map_test_csv, px_um_x=params['map_x_step'], px_um_y= params['map_y_step'], out_path=tiff_filename_csv)

#         # Create the output TIFF filename
#         tiff_filename = os.path.join(map_path, tiff_folder +"tdms_test_w_correction_v2" + ".tiff")
#         print(map_path)

#         # save tiff file 
#         if map_test is not None:
#             _ = maps.save_map_as_tiff (map_test, px_um_x=params['map_x_step'], px_um_y= params['map_y_step'], out_path=tiff_filename)

#     except Exception as e:
#         print(f"Error processing {map_path}: {e}")  
#         continue
        